## Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as `task_images` alongside the task prompt. The agent then processes these images throughout its execution.  

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.  

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.  

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [1]:
!pip install smolagents

In [2]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg"
]

images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    response = requests.get(url,headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

Now that we have the images, the agent will tell us wether the guests is actually a superhero (Wonder Woman) or a villian (The Joker).

In [6]:
import os
import smolagents
from smolagents import ToolCallingAgent, OpenAIServerModel
from google.colab import userdata

google_api_key = None

# 1. Try to fetch from Colab Secrets
try:
    google_api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    pass

# 2. Reset if the key contains code blocks or syntax by mistake
if google_api_key:
    bad_keywords = ["OpenAIServerModel", "api_key", "model_id", "os.environ", "def ", "import "]
    if any(keyword in google_api_key for keyword in bad_keywords):
        google_api_key = None

# 3. Prompt for the raw key if we do not have a clean one
if not google_api_key:
    import getpass
    print("Please enter your raw Google AI Studio API Key. Do not paste python code.")
    google_api_key = getpass.getpass("Google API Key: ").strip()

# Configure Gemini via Google AI Studio's OpenAI-compatible endpoint using gemini-3.8-flash
model = OpenAIServerModel(
    model_id="gemini-3.8-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=google_api_key
)

# Instantiate a ToolCallingAgent instead of a CodeAgent to cleanly receive text responses without syntax errors
agent = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=5,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing and return the description. │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
╰─ OpenAIModel - gemini-3.8-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
ChatCompletion(id='fUPNaqf9HJHJ-8YPq4ymQQ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None,    
message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None,          
function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_85883',                             
function=Function(arguments='{"answer":"The character shown in the photos is **The Joker**.\\n\\n### Makeup        
Description:\\n* **Skin:** Completely covered in chalk-white makeup across the face, neck, and ears, giving a      
bleached, ghostly appearance.\\n* **Mouth / Smile:** Bright red lipstick applied in an exaggerated, permanently    
wide, maniacal grin extending past the natural lip line, exposing teeth.\\n* **Eyes & Brows:** Thin, highly arched 
and sinister eyebrows with dark blue/black shading and liner around the eye sockets, creating an intense,          
theatrical gaze.\\n* **Hair:** Dyed vivid emerald-green, neatly combed and slicked back.\\n\\n### Costume          
Description:\\n* **First Image (Jack Nicholson\'s 1989 Portrayal):** \\n  * A textured, vibrant purple coat with   
wide lapels.\\n  * An ochre/mustard-yellow dress shirt with a pointed collar.\\n  * A glossy, wide purple satin    
cravat/ribbon tie tied at the collar.\\n* **Second Image (Comic Illustration by Alex Ross):** \\n  * A tailored    
dark suit/tuxedo jacket with a white flower (acid-squirting boutonnière) pinned to the lapel.\\n  * A crisp white  
collared dress shirt with a matching tie/vest.\\n  * Formal white gloves, holding a signature Joker playing        
card."}', name='final_answer'), type='function', extra_content={'google': {'thought_signature':                    
'EscQCsQQAWkUfROImpHuAyTUek9ll74ERZijNPg3/29Jilan6c3jXxMZJMIFulTfIucOIBwTtI6hm3QnXNWdtF1g3mo0UALEaXwd5sE7hbK9vqmHjG
[38;2;230;23

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "The character shown in the photos is **The             │
│ Joker**.\n\n### Makeup Description:\n* **Skin:** Completely covered in chalk-white makeup across the face,      │
│ neck, and ears, giving a bleached, ghostly appearance.\n* **Mouth / Smile:** Bright red lipstick applied in an  │
│ exaggerated, permanently wide, maniacal grin extending past the natural lip line, exposing teeth.\n* **Eyes &   │
│ Brows:** Thin, highly arched and sinister eyebrows with dark blue/black shading and liner around the eye        │
│ sockets, creating an intense, theatrical gaze.\n* **Hair:** Dyed vivid emerald-green, neatly combed and slicked │
│ back.\n\n### Costume Description:\n* **First Image (Jack Nicholson's 1989 Portrayal):** \n  * A textured,       │
│ vibrant purple coat with wide lapels.\n  * An ochre/mustard-yellow dress shirt with a pointed collar.\n  * A    │
│ glossy, wide purple satin cravat/ribbon tie tied at the collar.\n* **Second Image (Comic Illustration by Alex   │
│ Ross):** \n  * A tailored dark suit/tuxedo jacket with a white flower (acid-squirting boutonnière) pinned to    │
│ the lapel.\n  * A crisp white collared dress shirt with a matching tie/vest.\n  * Formal white gloves, holding  │
│ a signature Joker playing card."}                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The character shown in the photos is **The Joker**.

### Makeup Description:
* **Skin:** Completely covered in chalk-white makeup across the face, neck, and ears, giving a bleached, ghostly 
appearance.
* **Mouth / Smile:** Bright red lipstick applied in an exaggerated, permanently wide, maniacal grin extending past 
the natural lip line, exposing teeth.
* **Eyes & Brows:** Thin, highly arched and sinister eyebrows with dark blue/black shading and liner around the eye
sockets, creating an intense, theatrical gaze.
* **Hair:** Dyed vivid emerald-green, neatly combed and slicked back.

### Costume Description:
* **First Image (Jack Nicholson's 1989 Portrayal):** 
  * A textured, vibrant purple coat with wide lapels.
  * An ochre/mustard-yellow dress shirt with a pointed collar.
  * A glossy, wide purple satin cravat/ribbon tie tied at the collar.
* **Second Image (Comic Illustration by Alex Ross):** 
  * A tailored dark suit/tuxedo jacket with a white flower (acid-squirting boutonnière) pinned to the lapel.
  * A crisp white collared dress shirt with a matching tie/vest.
  * Formal white gloves, holding a signature Joker playing card.

Final answer: The character shown in the photos is **The Joker**.

### Makeup Description:
* **Skin:** Completely covered in chalk-white makeup across the face, neck, and ears, giving a bleached, ghostly 
appearance.
* **Mouth / Smile:** Bright red lipstick applied in an exaggerated, permanently wide, maniacal grin extending past 
the natural lip line, exposing teeth.
* **Eyes & Brows:** Thin, highly arched and sinister eyebrows with dark blue/black shading and liner around the eye
sockets, creating an intense, theatrical gaze.
* **Hair:** Dyed vivid emerald-green, neatly combed and slicked back.

### Costume Description:
* **First Image (Jack Nicholson's 1989 Portrayal):** 
  * A textured, vibrant purple coat with wide lapels.
  * An ochre/mustard-yellow dress shirt with a pointed collar.
  * A glossy, wide purple satin cravat/ribbon tie tied at the collar.
* **Second Image (Comic Illustration by Alex Ross):** 
  * A tailored dark suit/tuxedo jacket with a white flower (acid-squirting boutonnière) pinned to the lapel.
  * A crisp white collared dress shirt with a matching tie/vest.
  * Formal white gloves, holding a signature Joker playing card.

[Step 1: Duration 13.63 seconds| Input tokens: 3,174 | Output tokens: 298]

In [9]:
response

"The character shown in the photos is **The Joker**.\n\n### Makeup Description:\n* **Skin:** Completely covered in chalk-white makeup across the face, neck, and ears, giving a bleached, ghostly appearance.\n* **Mouth / Smile:** Bright red lipstick applied in an exaggerated, permanently wide, maniacal grin extending past the natural lip line, exposing teeth.\n* **Eyes & Brows:** Thin, highly arched and sinister eyebrows with dark blue/black shading and liner around the eye sockets, creating an intense, theatrical gaze.\n* **Hair:** Dyed vivid emerald-green, neatly combed and slicked back.\n\n### Costume Description:\n* **First Image (Jack Nicholson's 1989 Portrayal):** \n  * A textured, vibrant purple coat with wide lapels.\n  * An ochre/mustard-yellow dress shirt with a pointed collar.\n  * A glossy, wide purple satin cravat/ribbon tie tied at the collar.\n* **Second Image (Comic Illustration by Alex Ross):** \n  * A tailored dark suit/tuxedo jacket with a white flower (acid-squirting 